# ⚡ Spark — Kaggle Edition
Trained **from scratch** (no pretrained weights). Same stage system as the Colab version, plus:
- **Auto-resume**: if a session ends mid-training, run again and it picks up where it left off
- **Time guard**: stops and saves safely before Kaggle's session limit
- **Less padding waste**: faster training

## One-time setup
1. kaggle.com → sign in → verify your phone number (Settings) so GPUs unlock.
2. **Create dataset** (left menu → Datasets → New): upload your `.jsonl` data files (`conversation.jsonl`, etc.), name it e.g. `spark-data`.
3. Create a **new notebook**, then File → Import notebook → upload this file.
4. Right panel → **Session options**: Accelerator = **GPU T4 x2** (pick T4, not P100), Internet = **On**.
5. **Add Input** → add your `spark-data` dataset.

## Running
- Set `STAGE` in Cell 2, then **Run All** (or **Save Version → Save & Run All** to let it train in the background without keeping the page open).
- After a run: open the notebook's **Output** tab → "New Dataset" from the output (name it `spark-checkpoints`). Next session, add it as an input too — Spark restores tokenizer + checkpoints automatically.
- Only saved versions keep their output. Use **Save & Run All** for long runs.

## NEW: Pretraining stage (Spark 0.1)
Before chatting, Spark should *read* a lot of plain text so it learns how language works. Put raw text as **`.txt` files** in your dataset (books, articles, encyclopedia-style text; check licenses). Run with `STAGE = 'pretrain'`. This is the long one: it resumes across sessions, so just run it again after a 8-hour stop. Then set `STAGE = 'conversation'` (it starts from the pretrained Spark and replays text so it doesn't forget).


In [ ]:
# 1) SETUP
!pip -q install tokenizers
import torch, os, glob, shutil, json, math, random, time
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Accelerator to GPU T4!")
ROOT = '/kaggle/working/Spark'
for d in ['data', 'checkpoints', 'tokenizer']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

# Restore everything from any attached input (your data + previous checkpoints/tokenizer)
def restore_from_inputs():
    n = 0
    for path in glob.glob('/kaggle/input/**/*', recursive=True):
        if not os.path.isfile(path): continue
        name = os.path.basename(path)
        if name.endswith('.jsonl') or name.endswith('.txt'): dest = f'{ROOT}/data/{name}'
        elif name == 'tokenizer.json': dest = f'{ROOT}/tokenizer/{name}'
        elif name.startswith('spark_') and ('.pt' in name): dest = f'{ROOT}/checkpoints/{name}'
        else: continue
        if not os.path.exists(dest):
            if name.endswith('.txt'): os.symlink(path, dest)   # big text: link, don't copy
            else: shutil.copy(path, dest)
            n += 1; print("restored:", name)
    print(f"{n} files restored.")
restore_from_inputs()
print("data files:", os.listdir(f'{ROOT}/data'))

In [ ]:
# 2) CONFIG
from dataclasses import dataclass

@dataclass
class ModelConfig:
    vocab_size: int = 16000     # FIXED forever
    block_size: int = 512
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768           # ~110M params
    dropout: float = 0.1
cfg = ModelConfig()
# vocab_size, block_size and architecture must NEVER change between versions.

STAGES = {
    # kind='pretrain': raw .txt files (reads lots of text; learns language + general knowledge)
    # kind='chat'    : .jsonl conversations (learns how to talk / a specific skill)
    'pretrain':     dict(version='0.1', kind='pretrain', files='*.txt',                  steps=12000, lr=6e-4),
    'conversation': dict(version='0.2', kind='chat', files=['conversation.jsonl'],       steps=1500,  lr=1e-4),
    'math_8':       dict(version='0.3', kind='chat', files=['math_8.jsonl'],             steps=2000,  lr=1e-4),
    'math_hs':      dict(version='0.4', kind='chat', files=['math_hs.jsonl'],            steps=2000,  lr=1e-4),
    'code_python':  dict(version='0.5', kind='chat', files=['code_python.jsonl'],        steps=3000,  lr=1e-4),
}
STAGE_ORDER = list(STAGES)

STAGE = 'pretrain'        # <-- change this each update: pretrain -> conversation -> math_8 -> ...
MODE = 'add'              # 'add' (keep knowledge) | 'replace' (new data only, keep weights) | 'fresh' (random start)
REPLAY_FRACTION = 0.35
BATCH, ACCUM = 16, 4      # out of memory? use BATCH=8, ACCUM=8
TIME_LIMIT_H = 8.0        # safe stop + save before Kaggle's session limit
FORCE_RETRAIN = False     # True = retrain this stage even if its checkpoint exists

In [ ]:
# 3) MODEL
import torch.nn as nn, torch.nn.functional as F

class Attn(nn.Module):
    def __init__(s, c):
        super().__init__()
        s.qkv = nn.Linear(c.n_embd, 3*c.n_embd, bias=False)
        s.proj = nn.Linear(c.n_embd, c.n_embd, bias=False)
        s.h, s.drop = c.n_head, c.dropout
    def forward(s, x):
        B, T, C = x.shape
        q, k, v = s.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, s.h, C//s.h).transpose(1, 2) for t in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=s.drop if s.training else 0)
        return s.proj(y.transpose(1, 2).contiguous().view(B, T, C))

class Block(nn.Module):
    def __init__(s, c):
        super().__init__()
        s.n1, s.n2 = nn.LayerNorm(c.n_embd), nn.LayerNorm(c.n_embd)
        s.attn = Attn(c)
        s.mlp = nn.Sequential(nn.Linear(c.n_embd, 4*c.n_embd), nn.GELU(),
                              nn.Linear(4*c.n_embd, c.n_embd), nn.Dropout(c.dropout))
    def forward(s, x):
        x = x + s.attn(s.n1(x))
        return x + s.mlp(s.n2(x))

class Spark(nn.Module):
    def __init__(s, c):
        super().__init__()
        s.c = c
        s.tok = nn.Embedding(c.vocab_size, c.n_embd)
        s.pos = nn.Embedding(c.block_size, c.n_embd)
        s.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        s.nf = nn.LayerNorm(c.n_embd)
        s.head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        s.head.weight = s.tok.weight
        s.apply(s._init)
    def _init(s, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
    def forward(s, idx, targets=None):
        B, T = idx.shape
        x = s.tok(idx) + s.pos(torch.arange(T, device=idx.device))
        for b in s.blocks: x = b(x)
        logits = s.head(s.nf(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss
    @torch.no_grad()
    def generate(s, idx, max_new=200, temp=0.8, top_k=40, stop_id=None):
        for _ in range(max_new):
            logits, _ = s(idx[:, -s.c.block_size:])
            logits = logits[:, -1] / max(temp, 1e-5)
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('inf')
            nxt = torch.multinomial(F.softmax(logits, -1), 1)
            idx = torch.cat([idx, nxt], 1)
            if stop_id is not None and nxt.item() == stop_id: break
        return idx

device = 'cuda'
print("Params (M):", sum(p.numel() for p in Spark(cfg).parameters())/1e6)

In [ ]:
# 4) DATA: handles BOTH formats, multi-turn chats, removes exact duplicates
SPECIAL = ['<pad>', '<eos>', '<user>', '<spark>']   # fixed forever

def load_jsonl(fname):
    """Returns a list of conversations; each is a list of (role, text)."""
    path = f'{ROOT}/data/{fname}'
    assert os.path.exists(path), f"Missing {fname}! Add it to your Kaggle dataset and re-attach it as an input."
    convs, seen, dupes = [], set(), 0
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            d = json.loads(line)
            if 'messages' in d:
                conv = [(m['role'], m['content']) for m in d['messages']]
            else:
                conv = [('user', d['prompt']), ('assistant', d['response'])]
            key = json.dumps(conv)
            if key in seen: dupes += 1; continue
            seen.add(key); convs.append(conv)
    print(f"{fname}: {len(convs)} conversations ({dupes} exact duplicates removed)")
    return convs

def encode_conv(tk, conv):
    """User turns are context only; the model learns just the assistant replies."""
    eos = tk.token_to_id('<eos>')
    ids, labels = [], []
    for role, text in conv:
        if role == 'user':
            t = tk.encode(f"<user>{text}<spark>").ids
            ids += t; labels += [-100]*len(t)
        else:
            t = tk.encode(text).ids + [eos]
            ids += t; labels += t
    return ids[:cfg.block_size], labels[:cfg.block_size]

def make_batch(pools, weights, bs):
    exs = [pick(random.choices(pools, weights)[0]) for _ in range(bs)]
    L = max(2, max(len(e[0]) for e in exs))
    X = torch.zeros(bs, L, dtype=torch.long)
    Y = torch.full((bs, L), -100, dtype=torch.long)
    for i, (ids, lab) in enumerate(exs):
        X[i, :len(ids)] = torch.tensor(ids); Y[i, :len(lab)] = torch.tensor(lab)
    return X[:, :-1].to(device), Y[:, 1:].to(device)


# ---------- raw-text pretraining support ----------
import numpy as np

class Stream:
    """All pretraining text as one long token array; training samples random windows from it."""
    def __init__(self, arr): self.arr = arr
    def sample(self):
        n = cfg.block_size
        i = random.randint(0, len(self.arr) - n - 1)
        ids = self.arr[i:i+n].astype(np.int64).tolist()
        return ids, ids              # every token is trained on

def pick(pool):
    return pool.sample() if isinstance(pool, Stream) else random.choice(pool)

def text_files(): return sorted(glob.glob(f'{ROOT}/data/*.txt'))

def iter_text_chunks(max_chars=None, chunk=1_000_000):
    done = 0
    for fn in text_files():
        with open(fn, encoding='utf-8', errors='ignore') as f:
            while True:
                s = f.read(chunk)
                if not s: break
                yield s; done += len(s)
                if max_chars and done >= max_chars: return

def build_stream(tk):
    eos = tk.token_to_id('<eos>'); parts, batch = [], []
    def flush():
        for enc in tk.encode_batch(batch): parts.append(np.array(enc.ids, dtype=np.uint16))
        batch.clear()
    for fn in text_files():
        for s in iter_text_chunks_one(fn):
            batch.append(s)
            if len(batch) >= 8: flush()
        if batch: flush()
        parts.append(np.array([eos], dtype=np.uint16))     # marks the end of each file
    arr = np.concatenate(parts)
    print(f"Pretraining text: {len(arr)/1e6:.1f}M tokens from {len(text_files())} file(s)")
    return arr

def iter_text_chunks_one(fn, chunk=1_000_000):
    with open(fn, encoding='utf-8', errors='ignore') as f:
        while True:
            s = f.read(chunk)
            if not s: break
            yield s

def make_pool(tk, stage):
    """Returns (training pool, held-out examples) for any stage, pretrain or chat."""
    st = STAGES[stage]
    if st['kind'] == 'pretrain':
        arr = build_stream(tk)
        assert len(arr) > cfg.block_size * 200, "Not enough text. Add .txt files to your dataset (Add Input)."
        n_val = max(len(arr)//100, cfg.block_size * 64)
        train, val = Stream(arr[:-n_val]), Stream(arr[-n_val:])
        return train, [val.sample() for _ in range(64)]
    convs = [encode_conv(tk, c) for fn in st['files'] for c in load_jsonl(fn)]
    random.Random(0).shuffle(convs)
    n_val = max(20, len(convs)//30)
    print(f"{stage}: {len(convs)-n_val} training conversations | {n_val} held out")
    return convs[n_val:], convs[:n_val]


In [ ]:
# 5) TOKENIZER (trained once on ALL data files, then frozen; restored automatically in later sessions)
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
TOK_PATH = f'{ROOT}/tokenizer/tokenizer.json'

if os.path.exists(TOK_PATH):
    tk = Tokenizer.from_file(TOK_PATH); print("Loaded existing tokenizer (frozen).")
else:
    texts = []
    for fn in glob.glob(f'{ROOT}/data/*.jsonl'):
        for conv in load_jsonl(os.path.basename(fn)): texts += [t for _, t in conv]
    n_chat = len(texts)
    texts += list(iter_text_chunks(max_chars=150_000_000))   # raw text, capped to keep memory sane
    print(f"Tokenizer data: {n_chat} chat texts + {len(texts)-n_chat} text chunks")
    assert texts, "No .jsonl or .txt data found. Attach your dataset as an Input (Add Input)."
    tk = Tokenizer(models.BPE())
    tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tk.decoder = decoders.ByteLevel()
    tr = trainers.BpeTrainer(vocab_size=cfg.vocab_size, special_tokens=SPECIAL,
                             initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tk.train_from_iterator(texts, tr)
    tk.save(TOK_PATH); print("Trained + saved tokenizer. Vocab:", tk.get_vocab_size())

In [ ]:
# 6) TRAIN A STAGE (auto-resumes; stops safely before the session limit)
def ckpt_path(stage): return f"{ROOT}/checkpoints/spark_{STAGES[stage]['version']}_{stage}.pt"
def safe_save(obj, path):
    torch.save(obj, path + '.tmp'); os.replace(path + '.tmp', path)

@torch.no_grad()
def eval_loss(model, val):
    model.eval(); tot = n = 0
    for i in range(0, len(val), BATCH):
        b = val[i:i+BATCH]; L = max(2, max(len(e[0]) for e in b))
        X = torch.zeros(len(b), L, dtype=torch.long); Y = torch.full((len(b), L), -100, dtype=torch.long)
        for j, (ids, lab) in enumerate(b):
            X[j, :len(ids)] = torch.tensor(ids); Y[j, :len(lab)] = torch.tensor(lab)
        with torch.autocast('cuda', dtype=torch.float16):
            _, l = model(X[:, :-1].to(device), Y[:, 1:].to(device))
        tot += l.item(); n += 1
    model.train(); return tot / max(n, 1)

def train_stage(stage, mode=MODE):
    t0 = time.time()
    st = STAGES[stage]; i = STAGE_ORDER.index(stage)
    prev = STAGE_ORDER[i-1] if i > 0 else None
    final, part = ckpt_path(stage), ckpt_path(stage) + '.partial'

    if os.path.exists(final) and not os.path.exists(part) and not FORCE_RETRAIN:
        print(f"Stage '{stage}' already trained ({final}). Set FORCE_RETRAIN=True to redo."); return
    model = Spark(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=st['lr'], betas=(0.9, 0.95), weight_decay=0.1)
    scaler = torch.amp.GradScaler('cuda')
    start = 0
    if os.path.exists(part):
        ck = torch.load(part, map_location=device)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); start = ck['step']
        print(f"RESUMING '{stage}' from step {start}")
    elif mode != 'fresh' and prev:
        assert os.path.exists(ckpt_path(prev)), f"Need previous stage checkpoint: {ckpt_path(prev)} (attach your spark-checkpoints dataset)"
        model.load_state_dict(torch.load(ckpt_path(prev), map_location=device)['model'])
        print(f"Starting from {prev} (v{STAGES[prev]['version']})")
    else:
        print("Starting from random weights.")

    new_pool, val_pool = make_pool(tk, stage)
    pools, weights = [new_pool], [1.0]
    if mode == 'add' and prev:
        old_pools = [make_pool(tk, s)[0] for s in STAGE_ORDER[:i]]    # replay every earlier stage
        pools = [new_pool] + old_pools
        weights = [1 - REPLAY_FRACTION] + [REPLAY_FRACTION/len(old_pools)]*len(old_pools)
        print(f"Replaying {len(old_pools)} earlier stage(s) in {REPLAY_FRACTION:.0%} of batches")

    steps, warm = st['steps'], 200
    model.train()
    for step in range(start, steps):
        lr = st['lr']*(step+1)/warm if step < warm else st['lr']*0.5*(1+math.cos(math.pi*(step-warm)/(steps-warm)))
        for g in opt.param_groups: g['lr'] = max(lr, st['lr']*0.05)
        for _ in range(ACCUM):
            x, y = make_batch(pools, weights, BATCH)
            with torch.autocast('cuda', dtype=torch.float16):
                _, loss = model(x, y)
            scaler.scale(loss/ACCUM).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        if step % 100 == 0:
            print(f"step {step}/{steps}  loss {loss.item():.3f}  ({(time.time()-t0)/60:.1f} min)")
        if step % 500 == 499:
            print(f"   >>> held-out loss {eval_loss(model, val_pool):.3f}  (if this rises while train loss falls = memorizing)")
        out_of_time = (time.time() - t0)/3600 > TIME_LIMIT_H
        if (step % 500 == 499) or out_of_time:
            safe_save({'model': model.state_dict(), 'opt': opt.state_dict(), 'step': step+1}, part)
        if out_of_time:
            print(f"Time limit reached at step {step+1}. Progress saved. Start a new session and run again to RESUME.")
            return
    safe_save({'model': model.state_dict(), 'cfg': cfg.__dict__}, final)
    if os.path.exists(part): os.remove(part)
    print("DONE. Saved:", final)

train_stage(STAGE)

In [ ]:
# 7) CHAT (uses the finished checkpoint, or the partial one if training isn't finished)
def load_spark(stage):
    p = ckpt_path(stage)
    if not os.path.exists(p): p += '.partial'
    m = Spark(cfg).to(device)
    m.load_state_dict(torch.load(p, map_location=device)['model']); m.eval()
    return m

def ask(m, prompt, **kw):
    ids = torch.tensor([tk.encode(f"<user>{prompt}<spark>").ids], device=device)
    out = m.generate(ids, stop_id=tk.token_to_id('<eos>'), **kw)[0].tolist()
    return tk.decode(out[ids.shape[1]:]).replace('<eos>', '').strip()

spark = load_spark(STAGE)
for q in ["Hey!", "I'm bored, let's chat", "I had a rough day, can you talk with me?"]:
    print("You:", q, "\nSpark:", ask(spark, q), "\n")
# FORGETTING TEST: after each new stage, re-ask old prompts. If old skills got worse, raise REPLAY_FRACTION.